# Adding semantic tags to an sqlite table

This notebook adds semantic tags to adverbials in the Estonian Reference corpus. The semantic types are added to two sqlite database tables *spatial_obl* and *advmod* into a new column called *ekilex_tag*.

In [1]:
#imports
import sqlite3
import os
import re

In [2]:
def column_exists(cursor, table, column):
    cursor.execute(f"PRAGMA table_info({table})")
    return any(row[1] == column for row in cursor.fetchall())

In [3]:
# database file path
filename = "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\ressurssid\\rektsioonid\\katrin\\andmebaasifailid\\v33_koondkorpus_transaktsioonid.db"

# connecting with database
conn_db = sqlite3.connect(filename)
cursor_db = conn_db.cursor()

### Save word and semantic type to dict

In [4]:
def word_semtype_fun(directory_str):
    word_semtype = [] # tuples of semtype and word, i.e. (amount, aegsamini)
    
    directory = os.fsencode(directory_str)
    
    for file in os.listdir(directory):
        file_name = os.fsdecode(file)
        filepath = directory_str + '\\' + file_name
        semtype = re.findall(r"^(?:adv_)?(.+?)\.[^.]+$", file_name)
        with open(filepath, "r", encoding="utf-8") as f:
            words = f.read().splitlines()
            for word in words:
                word_semtype.append((semtype[0], word))  
    return word_semtype

In [5]:
directory_obl =  "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_location_by_context\\base_data\\ekilexist\\wordlists_obl"
word_semtype_obl = word_semtype_fun(directory_obl)

In [5]:
directory_advmod =  "C:\\Users\\kertu.saul\\OneDrive - Eesti Keele Instituut\\Dokumendid\\doktoritoo\\estnltk_syntax_repo_kloon\\physical_location_labelling\\physical_location_by_context\\base_data\\ekilexist\\wordlists_advmod"
word_semtype_adv = word_semtype_fun(directory_advmod)
word_semtype_adv

[('amount', 'aegsamini'),
 ('amount', 'ainumalt'),
 ('amount', 'ammendavalt'),
 ('amount', 'arvamata'),
 ('amount', 'aupoolest'),
 ('amount', 'aupärast'),
 ('amount', 'enam-vähem'),
 ('amount', 'esimeseks'),
 ('amount', 'esiteks'),
 ('amount', 'etemini'),
 ('amount', 'fantastiliselt'),
 ('amount', 'forsseeritult'),
 ('amount', 'haaravalt'),
 ('amount', 'halvemini'),
 ('amount', 'halvimini'),
 ('amount', 'hambuni'),
 ('amount', 'harvanähtavalt'),
 ('amount', 'hiiglamoodi'),
 ('amount', 'hinge põhjani'),
 ('amount', 'hingepõhjani'),
 ('amount', 'hinge põhjast'),
 ('amount', 'hingepõhjast'),
 ('amount', 'hirmpalju'),
 ('amount', 'hulga'),
 ('amount', 'hulganiselt'),
 ('amount', 'hulgim'),
 ('amount', 'hullu'),
 ('amount', 'hullult'),
 ('amount', 'hullumoodi'),
 ('amount', 'hullupööra'),
 ('amount', 'hullusti'),
 ('amount', 'häbemata'),
 ('amount', 'ilmama'),
 ('amount', 'imeharva'),
 ('amount', 'imehästi'),
 ('amount', 'imevähe'),
 ('amount', 'issanda'),
 ('amount', 'jalaga segada'),
 ('a

### Add semantic types to lemmas in the database

In [6]:
def semtype_to_db(table_name, semtypes, cursor, conn):
    # Step 1: add new column to database table
    if not column_exists(cursor, table_name, "ekilex_tag"):
        cursor.execute("ALTER TABLE " + table_name + " ADD COLUMN ekilex_tag TEXT")

    # Step 2: add word + semantic tag to temporary table
    cursor.execute("CREATE TEMP TABLE IF NOT EXISTS temp_updates (lemma TEXT PRIMARY KEY, ekilex_tag TEXT)")
    cursor.executemany("INSERT INTO temp_updates (ekilex_tag, lemma) VALUES (?, ?)", semtypes)

    # Step 3: Add temporary table info to database table
    #ps, pronouns are excluded for spatial obliques
    cursor.execute(f"""
        UPDATE {table_name}
        SET ekilex_tag = (SELECT ekilex_tag FROM temp_updates WHERE temp_updates.lemma = {table_name}.lemma)
        WHERE pos != 'P' AND EXISTS (SELECT 1 FROM temp_updates WHERE temp_updates.lemma = {table_name}.lemma)
    """)

    conn.commit()
    conn.close()

In [ ]:
semtype_to_db('spatial_obl', word_semtype_obl, cursor_db, conn_db)

In [7]:
semtype_to_db('advmod', word_semtype_adv, cursor_db, conn_db)